# Does Polymarket Anticipate Financial Markets?
## Granger Causality Test & Cross-Correlation

---

**Notebook 03** — Pillar 2 of the thesis: *temporal correlation between Polymarket sentiment and financial markets.*

In the [previous notebook (02)](./02_crowd_quality.ipynb), we showed that **the Polymarket crowd is well-calibrated**: its Brier Score is significantly better than chance, proving that prices reflect reliable probabilities.

Now the crucial question: **if the crowd is reliable, does it move BEFORE financial markets?**

In other words: do Polymarket price movements contain *predictive* information about traditional financial asset movements (bonds, equities, commodities, crypto)?

**Method:**
1. **Temporal alignment** — merge Polymarket series (24/7) with financial data (limited trading sessions)
2. **Granger test** — test whether Polymarket returns *Granger-cause* financial returns
3. **Cross-correlation** — identify the optimal lag between the two series

In [ ]:
# ── Setup & Imports ──────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import correlate
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

plt.style.use("seaborn-v0_8-whitegrid")

# Project palette
BLUE = "#2196F3"
GREEN = "#4CAF50"
ORANGE = "#FF9800"
RED = "#F44336"
PURPLE = "#9C27B0"
GRAY = "#9E9E9E"

# Category → financial tickers mapping
CATEGORY_TICKERS = {
    "fed_rates":         ["TLT", "^TNX", "DX-Y.NYB", "SPY"],
    "inflation_cpi":     ["TLT", "GLD", "SPY", "^TNX"],
    "macro_gdp":         ["SPY", "QQQ", "^VIX"],
    "geopolitics_trade": ["SPY", "CL=F", "GLD", "DX-Y.NYB"],
    "crypto_price":      ["BTC-USD", "ETH-USD", "SOL-USD"],
}

print("Setup OK ✓")

## 1. Loading the Data

Four files produced by previous pipeline steps:
- **`selected_markets.csv`** — selected Polymarket markets (category, real outcome, matched financial ticker)
- **`polymarket_hourly.parquet`** — hourly VWAP series per market
- **`financial_hourly.csv`** — hourly financial prices (14 tickers, 2 years)
- **`financial_daily.csv`** — daily financial prices

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
# On Colab: upload files or mount GCS bucket polymarket-data-raw
# from google.colab import files
# uploaded = files.upload()

DATA_DIR = "../data"  # adjust if running from Colab

markets = pd.read_csv(f"{DATA_DIR}/selected_markets.csv")
poly    = pd.read_parquet(f"{DATA_DIR}/polymarket_hourly.parquet")
fin_h   = pd.read_csv(f"{DATA_DIR}/financial_hourly.csv")
fin_d   = pd.read_csv(f"{DATA_DIR}/financial_daily.csv")

# Normalize all datetimes to UTC
poly["hour_utc"]   = pd.to_datetime(poly["hour_utc"], utc=True)
fin_h["datetime"]  = pd.to_datetime(fin_h["datetime"], utc=True)
fin_d["datetime"]  = pd.to_datetime(fin_d["datetime"], utc=True)
markets["end_date"] = pd.to_datetime(markets["end_date"], format="ISO8601", utc=True)

print(f"Selected markets     : {len(markets)}")
print(f"Polymarket hourly    : {len(poly):,} obs, {poly['condition_id'].nunique()} markets")
print(f"Financial hourly     : {len(fin_h):,} obs, {fin_h['ticker'].nunique()} tickers")
print(f"Financial daily      : {len(fin_d):,} obs")
print(f"\nCategories : {markets['category'].value_counts().to_dict()}")
print(f"Financial tickers    : {sorted(fin_h['ticker'].unique())}")

## 2. The Temporal Alignment Problem

This is THE technical challenge of this analysis.

**Polymarket** trades **24/7** — a "Fed rate hike" market can move at 3 AM UTC on a Sunday.

**Traditional financial markets** have limited sessions:
- NYSE: 9:30 AM – 4:00 PM EST (2:30 PM – 9:00 PM UTC)
- No trading on weekends or holidays

**Crypto** (BTC, ETH, SOL) trades 24/7 like Polymarket — no alignment issue.

**Solution**: `pd.merge_asof()` with `direction="backward"` and a tolerance of **2 hours**.

For each Polymarket hourly data point, we find the most recent financial price within a 2h window. If no price exists (weekends, overnight), the observation is dropped.

In [ ]:
# ── Concrete alignment example ────────────────────────────────────────────────
# Pick one market and one ticker to visualize the merge_asof

# Enrich poly with market metadata
poly = poly.merge(
    markets[["condition_id", "category", "matched_tickers", "outcome_real", "end_date", "question"]],
    on="condition_id",
    how="left",
)

# Choose an example: first market with a TLT or SPY ticker
example_market = poly[poly["matched_tickers"].str.contains("TLT|SPY", na=False)].iloc[0]
example_cid = example_market["condition_id"]
example_ticker = "TLT" if "TLT" in str(example_market["matched_tickers"]) else "SPY"

poly_ex = poly[poly["condition_id"] == example_cid][["hour_utc", "vwap_yes"]].sort_values("hour_utc").head(72)
fin_ex = fin_h[fin_h["ticker"] == example_ticker][["datetime", "close"]].sort_values("datetime")

print(f"Example market : {example_market['question'][:80]}...")
print(f"Matched ticker : {example_ticker}")
print(f"\nPolymarket : {len(poly_ex)} hourly points (72h extract)")
print(f"  range: {poly_ex['hour_utc'].min()} to {poly_ex['hour_utc'].max()}")
print(f"\nFinancial  : {len(fin_ex):,} hourly points total")
print(f"  note the gaps: weekends and overnight (no NYSE trading)")

## 3. Full Hourly Alignment

We align **each pair (Polymarket market, financial ticker)** via `merge_asof`. For each market, we use the tickers defined in the `CATEGORY_TICKERS` mapping (via the `matched_tickers` column).

We then compute **returns** over multiple horizons:
- `fin_return_1h/4h/1d`: percentage change of the financial price
- `poly_return_1h/4h`: absolute change of the Polymarket VWAP (in probability points)

In [ ]:
# ── Hourly alignment ──────────────────────────────────────────────────────────

HOURLY_TOLERANCE = pd.Timedelta("2h")


def align_hourly(poly, fin_h):
    """For each (market, matched ticker), align via merge_asof."""
    results = []
    for cid, group in poly.groupby("condition_id"):
        tickers = group["matched_tickers"].iloc[0]
        if pd.isna(tickers):
            continue
        group = group.sort_values("hour_utc")

        for ticker in tickers.split(","):
            ticker = ticker.strip()
            fin_ticker = fin_h[fin_h["ticker"] == ticker][["datetime", "close"]].copy()
            fin_ticker = fin_ticker.sort_values("datetime")

            if fin_ticker.empty:
                continue

            merged = pd.merge_asof(
                group[["hour_utc", "condition_id", "vwap_yes", "volume_usd",
                        "category", "outcome_real", "end_date", "question"]],
                fin_ticker.rename(columns={"datetime": "hour_utc", "close": "fin_price"}),
                on="hour_utc",
                direction="backward",
                tolerance=HOURLY_TOLERANCE,
            )
            merged = merged.dropna(subset=["fin_price"])

            if merged.empty:
                continue

            merged["ticker"] = ticker
            merged["fin_return_1h"] = merged["fin_price"].pct_change(1)
            merged["fin_return_4h"] = merged["fin_price"].pct_change(4)
            merged["fin_return_1d"] = merged["fin_price"].pct_change(24)
            merged["poly_return_1h"] = merged["vwap_yes"].diff(1)
            merged["poly_return_4h"] = merged["vwap_yes"].diff(4)

            results.append(merged)

    return pd.concat(results, ignore_index=True) if results else pd.DataFrame()


# Run alignment
aligned = align_hourly(poly, fin_h)

print(f"Aligned observations : {len(aligned):,}")
print(f"Markets : {aligned['condition_id'].nunique()}")
print(f"Tickers : {aligned['ticker'].nunique()}")
print(f"\nBreakdown by category:")
for cat, g in aligned.groupby("category"):
    n_m = g["condition_id"].nunique()
    n_t = g["ticker"].nunique()
    print(f"  {cat:20s} | {n_m:3d} markets | {len(g):>8,} obs | {n_t} tickers")

## 4. Exploratory Visualization

Before running statistical tests, let's visually inspect a few aligned pairs. Left axis shows the Polymarket price (probability), right axis shows the financial price.

In [ ]:
# ── Aligned series: visual examples ───────────────────────────────────────────

# Select 3 interesting pairs (1 per category if possible)
example_pairs = (
    aligned.groupby(["condition_id", "ticker"])
    .agg(n=("hour_utc", "count"), category=("category", "first"), question=("question", "first"))
    .reset_index()
    .sort_values("n", ascending=False)
)

# Pick the 3 pairs with the most observations, different categories
seen_cats = set()
selected_pairs = []
for _, row in example_pairs.iterrows():
    if row["category"] not in seen_cats and len(selected_pairs) < 3:
        selected_pairs.append(row)
        seen_cats.add(row["category"])

fig, axes = plt.subplots(len(selected_pairs), 1, figsize=(12, 4 * len(selected_pairs)))
if len(selected_pairs) == 1:
    axes = [axes]

for idx, pair in enumerate(selected_pairs):
    ax1 = axes[idx]
    sub = aligned[(aligned["condition_id"] == pair["condition_id"]) &
                  (aligned["ticker"] == pair["ticker"])].sort_values("hour_utc")

    # Left axis: Polymarket
    ax1.plot(sub["hour_utc"], sub["vwap_yes"], color=BLUE, linewidth=1.5, label="Polymarket (VWAP)")
    ax1.set_ylabel("Polymarket Probability", color=BLUE, fontsize=11)
    ax1.tick_params(axis="y", labelcolor=BLUE)

    # Right axis: financial price
    ax2 = ax1.twinx()
    ax2.plot(sub["hour_utc"], sub["fin_price"], color=ORANGE, linewidth=1.5, alpha=0.8, label=pair["ticker"])
    ax2.set_ylabel(f"{pair['ticker']} Price", color=ORANGE, fontsize=11)
    ax2.tick_params(axis="y", labelcolor=ORANGE)

    title = f"{pair['question'][:70]}...\n[{pair['category']}] -> {pair['ticker']} ({pair['n']} obs)"
    ax1.set_title(title, fontsize=11, pad=10)
    ax1.grid(True, alpha=0.3)

    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=9)

fig.suptitle("Aligned Series: Polymarket vs Financial Markets", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 5. What is the Granger Causality Test?

The **Granger causality test** answers a simple question:

> *Do past values of X improve the prediction of Y, beyond what past values of Y alone allow?*

**Important**: this is **NOT** causality in the philosophical sense. It is **predictive causality** — a test of temporal lead.

Concretely:
- **H0** (null hypothesis): past Polymarket returns provide no additional information for predicting financial returns
- **If p < 0.05** → we reject H0 → Polymarket **statistically leads** the financial market
- We test multiple **lags** (1h to 12h) — the lag with the smallest p-value indicates the optimal prediction horizon

**Prerequisite**: the series must be **stationary** (no trend). That's why we work on *returns* (changes), not raw prices.

## 6. Stationarity Check (ADF Test)

Before applying the Granger test, we verify that the return series are stationary using the **Augmented Dickey-Fuller** test. A p-value < 0.05 means the series is stationary.

In [ ]:
# ── Statistical functions ─────────────────────────────────────────────────────

# Parameters
MAX_GRANGER_LAG = 12   # lags to test (in hours)
MAX_XCORR_LAG = 48     # lags for cross-correlation
MIN_OBS = 100          # minimum observations for a valid test


def check_stationarity(series):
    """ADF stationarity test. Returns (is_stationary, p_value)."""
    clean = series.dropna()
    if len(clean) < 20:
        return False, 1.0
    try:
        result = adfuller(clean, autolag="AIC")
        return result[1] < 0.05, result[1]
    except Exception:
        return False, 1.0


# Test stationarity on each (category, ticker) pair
stationarity_results = []
for (cat, ticker), g in aligned.groupby(["category", "ticker"]):
    poly_stat, poly_p = check_stationarity(g["poly_return_1h"])
    fin_stat, fin_p = check_stationarity(g["fin_return_1h"])
    stationarity_results.append({
        "Category": cat,
        "Ticker": ticker,
        "N obs": len(g),
        "Poly stationary": "Yes" if poly_stat else "No",
        "Poly p-value": f"{poly_p:.4f}",
        "Fin stationary": "Yes" if fin_stat else "No",
        "Fin p-value": f"{fin_p:.4f}",
    })

stat_df = pd.DataFrame(stationarity_results)
print("ADF stationarity test on 1h returns:")
print("(p < 0.05 = stationary = OK for Granger)\n")
print(stat_df.to_string(index=False))

## 7. Granger Causality Test

We aggregate all markets within a category into a **single signal** (volume-weighted average), then test whether this signal Granger-causes financial returns.

Volume weighting gives more weight to liquid markets — where the information is most reliable.

In [ ]:
# ── Granger test + Cross-correlation ──────────────────────────────────────────


def run_granger_test(poly_returns, fin_returns, max_lag=MAX_GRANGER_LAG):
    """Granger causality test. Returns dict {lag: p_value}."""
    combined = pd.DataFrame({"fin": fin_returns, "poly": poly_returns}).dropna()
    if len(combined) < MIN_OBS:
        return {}
    try:
        results = grangercausalitytests(combined[["fin", "poly"]], maxlag=max_lag, verbose=False)
        return {lag: result[0]["ssr_ftest"][1] for lag, result in results.items()}
    except Exception:
        return {}


def cross_correlation(poly_returns, fin_returns, max_lag=MAX_XCORR_LAG):
    """Normalized cross-correlation. Lag > 0 = Polymarket leads."""
    combined = pd.DataFrame({"poly": poly_returns, "fin": fin_returns}).dropna()
    if len(combined) < MIN_OBS:
        return np.array([]), np.array([])

    p = (combined["poly"] - combined["poly"].mean()) / combined["poly"].std()
    f = (combined["fin"] - combined["fin"].mean()) / combined["fin"].std()

    xcorr = correlate(p, f, mode="full") / len(combined)
    center = len(f) - 1
    start = max(center - max_lag, 0)
    end = min(center + max_lag + 1, len(xcorr))
    lags = np.arange(start - center, end - center)
    corrs = xcorr[start:end]
    return lags, corrs


def analyze_category_ticker(df, category, ticker):
    """Full analysis for a (category, ticker) pair."""
    sub = df[(df["category"] == category) & (df["ticker"] == ticker)].copy()
    if sub.empty:
        return None

    # Aggregate by hour: volume-weighted average
    agg = (
        sub.sort_values("hour_utc")
        .groupby("hour_utc")
        .agg(
            poly_return=("poly_return_1h", lambda x: np.average(
                x, weights=sub.loc[x.index, "volume_usd"].clip(lower=1))),
            fin_return=("fin_return_1h", "mean"),
            n_markets=("condition_id", "nunique"),
        )
        .dropna()
    )

    if len(agg) < MIN_OBS:
        return None

    # Stationarity
    poly_stat, poly_p = check_stationarity(agg["poly_return"])
    fin_stat, fin_p = check_stationarity(agg["fin_return"])

    # Granger
    granger_pvals = run_granger_test(agg["poly_return"], agg["fin_return"])

    # Cross-correlation
    lags, xcorr = cross_correlation(agg["poly_return"], agg["fin_return"])

    # Best lag
    best_lag, best_corr = None, 0
    peak_lag_positive, peak_corr_positive = None, 0

    if len(lags) > 0:
        best_idx = np.argmax(np.abs(xcorr))
        best_lag = int(lags[best_idx])
        best_corr = float(xcorr[best_idx])

        pos_mask = lags > 0
        if pos_mask.any():
            pos_idx = np.argmax(np.abs(xcorr[pos_mask]))
            peak_lag_positive = int(lags[pos_mask][pos_idx])
            peak_corr_positive = float(xcorr[pos_mask][pos_idx])

    min_granger_p = min(granger_pvals.values()) if granger_pvals else 1.0
    best_granger_lag = min(granger_pvals, key=granger_pvals.get) if granger_pvals else None

    return {
        "category": category,
        "ticker": ticker,
        "n_obs": len(agg),
        "n_markets": sub["condition_id"].nunique(),
        "poly_stationary": poly_stat,
        "fin_stationary": fin_stat,
        "granger_min_p": min_granger_p,
        "granger_best_lag": best_granger_lag,
        "granger_significant": min_granger_p < 0.05,
        "xcorr_best_lag": best_lag,
        "xcorr_best_corr": best_corr,
        "xcorr_poly_lead_lag": peak_lag_positive,
        "xcorr_poly_lead_corr": peak_corr_positive,
        "granger_all_pvals": granger_pvals,
        "xcorr_lags": lags,
        "xcorr_values": xcorr,
    }


# ── Run on all pairs ─────────────────────────────────────────────────────────

pairs = aligned.groupby(["category", "ticker"]).size().reset_index(name="n")
pairs = pairs[pairs["n"] >= MIN_OBS]
print(f"Pairs to test: {len(pairs)}\n")

results = []
for _, row in pairs.iterrows():
    cat, ticker = row["category"], row["ticker"]
    result = analyze_category_ticker(aligned, cat, ticker)
    if result:
        sig = "✓" if result["granger_significant"] else "✗"
        print(f"  {cat:20s} -> {ticker:12s} | p={result['granger_min_p']:.4f} {sig} | "
              f"lead={result['xcorr_poly_lead_lag']}h | n={result['n_obs']}")
        results.append(result)

print(f"\n{'─' * 60}")
sig_count = sum(1 for r in results if r["granger_significant"])
print(f"Results: {len(results)} pairs tested, {sig_count} significant (p < 0.05)")

## 8. Granger Results Heatmap

The key visualization: for each **(Polymarket category, financial ticker)** pair, the color indicates the Granger test p-value.

- **Dark green** (p < 0.05): Polymarket leads significantly
- **Red** (p > 0.10): no signal detected

In [ ]:
# ── Granger heatmap ───────────────────────────────────────────────────────────

categories = sorted(set(r["category"] for r in results))
tickers = sorted(set(r["ticker"] for r in results))

matrix = pd.DataFrame(index=categories, columns=tickers, dtype=float)
for r in results:
    matrix.loc[r["category"], r["ticker"]] = r["granger_min_p"]

fig, ax = plt.subplots(figsize=(12, 6))
matrix_vals = matrix.values.astype(float)

im = ax.imshow(matrix_vals, cmap="RdYlGn_r", vmin=0, vmax=0.15, aspect="auto")

ax.set_xticks(range(len(tickers)))
ax.set_xticklabels(tickers, rotation=45, ha="right", fontsize=10)
ax.set_yticks(range(len(categories)))
ax.set_yticklabels(categories, fontsize=10)

# Annotate each cell with the p-value
for i in range(len(categories)):
    for j in range(len(tickers)):
        val = matrix_vals[i, j]
        if np.isnan(val):
            text = "—"
            color = "gray"
        else:
            text = f"{val:.3f}"
            color = "white" if val < 0.05 else "black"
        ax.text(j, i, text, ha="center", va="center", fontsize=9, color=color,
                fontweight="bold" if not np.isnan(val) and val < 0.05 else "normal")

plt.colorbar(im, ax=ax, label="Granger p-value (green = significant)")
ax.set_title("Granger Causality Test: Does Polymarket Lead Financial Markets?",
             fontsize=13, pad=15)

fig.tight_layout()
fig.savefig("../outputs/granger_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved -> outputs/granger_heatmap.png")

## 9. Cross-Correlation

Cross-correlation complements the Granger test by showing **at which lag the correlation is maximized**:

- **Lag > 0** (blue bars): Polymarket moves **BEFORE** the financial market -> Polymarket **leads**
- **Lag < 0** (gray bars): the financial market moves before Polymarket
- **Lag = 0**: simultaneous movement

The red dashed line marks lag zero.

In [ ]:
# ── Cross-correlation plots ───────────────────────────────────────────────────

# Top 6 most significant pairs
significant = [r for r in results if r["granger_significant"] and len(r["xcorr_lags"]) > 0]
significant.sort(key=lambda r: r["granger_min_p"])
top = significant[:6]

if not top:
    # Fallback: take the 6 best even if not significant
    top = sorted([r for r in results if len(r["xcorr_lags"]) > 0],
                 key=lambda r: r["granger_min_p"])[:6]
    print("No significant pairs — showing the 6 best p-values")

n = len(top)
cols = min(3, n)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
if n == 1:
    axes = np.array([axes])
axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

for idx, r in enumerate(top):
    ax = axes[idx]
    lags = r["xcorr_lags"]
    xcorr = r["xcorr_values"]

    colors = [BLUE if l > 0 else GRAY for l in lags]
    ax.bar(lags, xcorr, color=colors, alpha=0.7, width=1.0)
    ax.axvline(x=0, color=RED, linestyle="--", alpha=0.5, linewidth=1.5)

    ax.set_title(f"{r['category']} -> {r['ticker']}\np = {r['granger_min_p']:.4f}",
                 fontsize=10, pad=8)
    ax.set_xlabel("Lag (hours)", fontsize=9)
    ax.set_ylabel("Correlation", fontsize=9)
    ax.grid(True, alpha=0.3)

# Hide empty axes
for idx in range(n, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle("Cross-Correlation: blue = Polymarket leads the financial market",
             fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig("../outputs/cross_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved -> outputs/cross_correlation.png")

## 10. Summary of Results

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────

summary_cols = ["category", "ticker", "n_obs", "n_markets",
                "granger_min_p", "granger_best_lag", "granger_significant",
                "xcorr_poly_lead_lag", "xcorr_poly_lead_corr"]

summary_df = pd.DataFrame([{k: r[k] for k in summary_cols} for r in results])
summary_df = summary_df.sort_values("granger_min_p")
summary_df.columns = ["Category", "Ticker", "N obs", "N markets",
                       "Min p-value", "Best lag (h)", "Significant",
                       "Poly lead (h)", "Lead corr"]

print("=" * 80)
print("SUMMARY TABLE — Granger Test: Polymarket -> Financial Markets")
print("=" * 80)
print()
print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Global stats
n_total = len(results)
n_sig = sum(1 for r in results if r["granger_significant"])
print(f"\n{'─' * 80}")
print(f"Pairs tested          : {n_total}")
print(f"Significant pairs     : {n_sig} ({100 * n_sig / n_total:.0f}%)" if n_total > 0 else "")

if n_sig > 0:
    top_results = sorted([r for r in results if r["granger_significant"]],
                         key=lambda r: r["granger_min_p"])
    print(f"\nTop pairs:")
    for r in top_results[:5]:
        print(f"  {r['category']:20s} -> {r['ticker']:12s} | "
              f"p={r['granger_min_p']:.4f} | lag={r['granger_best_lag']}h | "
              f"lead corr={r['xcorr_poly_lead_corr']:+.4f}")

In [ ]:
# ── Save results ──────────────────────────────────────────────────────────────

save_cols = ["category", "ticker", "n_obs", "n_markets",
             "poly_stationary", "fin_stationary",
             "granger_min_p", "granger_best_lag", "granger_significant",
             "xcorr_best_lag", "xcorr_best_corr",
             "xcorr_poly_lead_lag", "xcorr_poly_lead_corr"]

results_df = pd.DataFrame([{k: r[k] for k in save_cols} for r in results])
results_df.to_csv(f"{DATA_DIR}/granger_results.csv", index=False)
print(f"Results saved -> data/granger_results.csv")

## 11. Conclusion & Implications

### What we showed

Across the tested pairs between Polymarket categories and financial assets:
- The **Granger test** identifies cases where Polymarket price movements contain predictive information about future financial returns
- **Cross-correlation** confirms the direction and magnitude of the temporal lead

### Interpretation

The categories most likely to show a signal are those where:
1. **Information is clear and binary** (the Fed hikes or doesn't) — the prediction market aggregates information efficiently
2. **Volume is sufficient** — an illiquid market does not produce a reliable signal
3. **The link to the financial asset is direct** (Fed decision -> TLT bonds, crypto price -> BTC-USD)

### Methodological limitations

These results should be interpreted with caution:

- **Sample size**: the number of markets per category remains limited. A significant test on few pairs may be a false positive.
- **Survivorship bias**: we only test markets for which we have data — illiquid or quickly resolved markets are underrepresented.
- **Granger ≠ causation**: even a significant test does not prove that Polymarket *causes* the financial movement. A third variable (news, leak) could cause both.
- **Multiple testing**: testing multiple pairs without correction (Bonferroni, FDR) increases the false positive risk.
- **Market regime**: the relationship may be non-stationary — strong during stress, absent in normal conditions.

### Takeaway

These results, combined with the calibration proof from the previous notebook, suggest that **Polymarket aggregates information faster than traditional markets on certain macro categories**. The crowd putting real money on the line produces an actionable signal — at least as a leading sentiment indicator, if not as a trading signal.

---

*Notebook 03/05 — Polymarket as a Leading Indicator for Financial Markets*
*Data: BigQuery (SII-WANGZJ trades) + yfinance*